# Gold Layer — agg_restaurant_ranking (design 1.5, split credit)

One row per restaurant (`restaurant_id`, current version for name / address). **Full overwrite** daily (`CREATE OR REPLACE`, atomic): the score is non-additive, so it is always recomputed. Run **after** `Analytic/gold/checks`.

| Column | Definition |
| --- | --- |
| `video_count` | `SUM(mention_count)` |
| `credited_views` | `SUM(latest view_count × mention_weight)` (split credit: a "top 10 quán" video gives each 1/10 of its views) |
| `credited_views_30d` | `SUM(views_gained in the last 30 days × mention_weight)` |
| `like_rate` | `SUM(likes × weight) ÷ SUM(views × weight)` (never an average of per-video rates) |
| `ranking_score` | `0.7 × credited_views_30d + 0.3 × like_rate`, each min-max scaled 0–1 within the city (first version, tune later) |
| `rank_in_city` | rank by `ranking_score` within the city |
| `channel_count` | distinct YouTube channels that mention the restaurant (one channel can be a paid ad; several independent channels are harder to fake) |
| `google_rating_adjusted` | Bayesian average: `(rating × reviews + C × city mean) ÷ (reviews + C)`, `C = RATING_PRIOR_REVIEWS`, so 5.0 from 3 reviews does not beat 4.5 from 3,000 |
| `legit_score` | 0–100, weighted mean of the parts below over the parts that have data (`LEGIT_WEIGHTS`) |
| `legit_label` | `Legit` = `legit_score ≥ LEGIT_MIN_SCORE` **and** enough evidence (≥ `LEGIT_MIN_REVIEWS` Google reviews or ≥ 2 channels); `Ít dữ liệu` = < `LEGIT_MIN_REVIEWS` reviews and 1 channel; else `Bình thường` |

**legit_score parts (each 0–1):** `rating` = `(google_rating_adjusted − 3.5) ÷ 1.5` clipped · `reviews` = `log(1 + reviews) ÷ log(1 + max reviews in the city)` · `channels` = `min(channel_count, 5) ÷ 5` · `like_rate` and `views_30d` = the min-max scaled values of `ranking_score`. YouTube has no share count, so shares are not used.

Descriptive columns (`cuisine`, `price_band`, `google_rating`, `latitude`, ...) are copied from the current `dim_restaurant` version so the dashboard reads one table.

"Last 30 days" is relative to the latest snapshot day, so a rerun gives the same result.


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

AGG = f"{GOLD}.agg_restaurant_ranking"
WINDOW_DAYS = 30
W_VIEWS_30D, W_LIKE_RATE = 0.7, 0.3
# legit_score (first version, tune after looking at the top 10)
LEGIT_WEIGHTS = {"rating": 0.35, "reviews": 0.20, "channels": 0.20, "like_rate": 0.15, "views_30d": 0.10}
RATING_PRIOR_REVIEWS = 50   # C in the Bayesian average
LEGIT_MIN_SCORE = 55        # dev run 31/10: best = 66.9 (4.7★, 7,084 reviews); 70 gave no Legit at all
LEGIT_MIN_REVIEWS = 100
W = LEGIT_WEIGHTS

print(f"Gold source: {GOLD}.fact_video_restaurant_mention, fact_video_daily_snapshot, dim_restaurant")
print(f"Gold target: {AGG}")


In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {AGG}
    COMMENT 'Restaurant ranking (split credit) + legit_score, full overwrite daily. Non-additive: never sum ranking_score / legit_score.'
    AS
    WITH latest_day AS (
        SELECT MAX(snapshot_date_key) AS k,
               CAST(date_format(date_sub(to_date(CAST(MAX(snapshot_date_key) AS STRING), 'yyyyMMdd'), {WINDOW_DAYS - 1}), 'yyyyMMdd') AS INT) AS window_start
        FROM {GOLD}.fact_video_daily_snapshot
    ),
    video_latest AS (   -- semi-additive counts: take each video's last snapshot
        SELECT video_key, view_count, like_count
        FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY video_key ORDER BY snapshot_date_key DESC) AS rn
              FROM {GOLD}.fact_video_daily_snapshot)
        WHERE rn = 1
    ),
    video_30d AS (      -- additive gains: sum over the window
        SELECT f.video_key, SUM(COALESCE(f.views_gained, 0)) AS views_gained_30d
        FROM {GOLD}.fact_video_daily_snapshot f CROSS JOIN latest_day l
        WHERE f.snapshot_date_key BETWEEN l.window_start AND l.k
        GROUP BY f.video_key
    ),
    credited AS (
        SELECT d.restaurant_id,
               SUM(m.mention_count)                                  AS video_count,
               COUNT(DISTINCT dv.channel_id)                         AS channel_count,
               SUM(COALESCE(vl.view_count, 0) * m.mention_weight)    AS credited_views,
               SUM(COALESCE(v30.views_gained_30d, 0) * m.mention_weight) AS credited_views_30d,
               SUM(COALESCE(vl.like_count, 0) * m.mention_weight)    AS credited_likes,
               SUM(CASE WHEN vl.like_count IS NOT NULL THEN COALESCE(vl.view_count, 0) * m.mention_weight END) AS credited_views_with_likes
        FROM {GOLD}.fact_video_restaurant_mention m
        JOIN {GOLD}.dim_restaurant d ON d.restaurant_key = m.restaurant_key
        LEFT JOIN {GOLD}.dim_video dv ON dv.video_key = m.video_key AND dv.video_key <> -1
        LEFT JOIN video_latest vl ON vl.video_key = m.video_key
        LEFT JOIN video_30d v30   ON v30.video_key = m.video_key
        WHERE m.restaurant_key <> -1
        GROUP BY d.restaurant_id
    ),
    enriched AS (
        SELECT c.*, cur.restaurant_name, cur.address, cur.district, COALESCE(cur.city, 'Unknown') AS city,
               cur.latitude, cur.longitude, cur.cuisine, cur.cuisine_detail, cur.price_band, cur.price_min_vnd,
               cur.price_max_vnd, cur.google_rating, cur.google_rating_count, cur.google_maps_uri, cur.photo_name,
               c.credited_likes / NULLIF(c.credited_views_with_likes, 0) AS like_rate
        FROM credited c
        JOIN {GOLD}.dim_restaurant cur ON cur.restaurant_id = c.restaurant_id AND cur.is_current
    ),
    scaled AS (
        SELECT *,
            COALESCE((credited_views_30d - MIN(credited_views_30d) OVER w)
                     / NULLIF(MAX(credited_views_30d) OVER w - MIN(credited_views_30d) OVER w, 0), 0) AS views_30d_scaled,
            COALESCE((like_rate - MIN(like_rate) OVER w)
                     / NULLIF(MAX(like_rate) OVER w - MIN(like_rate) OVER w, 0), 0) AS like_rate_scaled
        FROM enriched
        WINDOW w AS (PARTITION BY city)
    ),
    rated AS (
        SELECT *,
            (google_rating * google_rating_count + {RATING_PRIOR_REVIEWS} * AVG(google_rating) OVER w)
                / (google_rating_count + {RATING_PRIOR_REVIEWS})                         AS google_rating_adjusted,
            ln(1 + google_rating_count) / NULLIF(ln(1 + MAX(google_rating_count) OVER w), 0) AS reviews_part
        FROM scaled
        WINDOW w AS (PARTITION BY city)
    ),
    parts AS (
        SELECT *,
            least(greatest((google_rating_adjusted - 3.5) / 1.5, 0), 1) AS rating_part,
            least(channel_count, 5) / 5.0                               AS channels_part
        FROM rated
    ),
    legit AS (   -- weighted mean over the parts that have data (no Google match -> YouTube parts only)
        SELECT *,
            100 * ( COALESCE({W['rating']} * rating_part, 0) + COALESCE({W['reviews']} * reviews_part, 0)
                  + {W['channels']} * channels_part
                  + CASE WHEN like_rate IS NOT NULL THEN {W['like_rate']} * like_rate_scaled ELSE 0 END
                  + {W['views_30d']} * views_30d_scaled )
                / ( CASE WHEN rating_part IS NOT NULL THEN {W['rating']} ELSE 0 END
                  + CASE WHEN reviews_part IS NOT NULL THEN {W['reviews']} ELSE 0 END
                  + {W['channels']}
                  + CASE WHEN like_rate IS NOT NULL THEN {W['like_rate']} ELSE 0 END
                  + {W['views_30d']} )                                  AS legit_score
        FROM parts
    )
    SELECT
        restaurant_id, restaurant_name, address, district, city, latitude, longitude,
        cuisine, cuisine_detail, price_band, price_min_vnd, price_max_vnd,
        google_rating, google_rating_count, round(google_rating_adjusted, 2) AS google_rating_adjusted,
        google_maps_uri, photo_name,
        CAST(video_count AS INT)                  AS video_count,
        CAST(channel_count AS INT)                AS channel_count,
        CAST(round(credited_views) AS BIGINT)     AS credited_views,
        CAST(round(credited_views_30d) AS BIGINT) AS credited_views_30d,
        like_rate,
        {W_VIEWS_30D} * views_30d_scaled + {W_LIKE_RATE} * like_rate_scaled AS ranking_score,
        CAST(RANK() OVER (PARTITION BY city ORDER BY {W_VIEWS_30D} * views_30d_scaled + {W_LIKE_RATE} * like_rate_scaled DESC,
                                                     credited_views DESC) AS INT) AS rank_in_city,
        round(legit_score, 1)                     AS legit_score,
        CASE
            WHEN legit_score >= {LEGIT_MIN_SCORE}
             AND (COALESCE(google_rating_count, 0) >= {LEGIT_MIN_REVIEWS} OR channel_count >= 2) THEN 'Legit'
            WHEN COALESCE(google_rating_count, 0) < {LEGIT_MIN_REVIEWS} AND channel_count < 2 THEN 'Ít dữ liệu'
            ELSE 'Bình thường'
        END                                       AS legit_label,
        CAST(RANK() OVER (PARTITION BY city ORDER BY legit_score DESC, credited_views DESC) AS INT) AS legit_rank_in_city,
        (SELECT k FROM latest_day)                AS as_of_date_key,
        current_timestamp()                       AS etl_date
    FROM legit
""")


In [ ]:
# --- Verify ---
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id) AS ids, COUNT(DISTINCT city) AS cities, MAX(as_of_date_key) AS as_of
    FROM {AGG}
""").collect()[0]
print(f"📊 {AGG}: {s['n']} restaurants ({s['ids']} distinct ids) in {s['cities']} city(ies), as of {s['as_of']} "
      + ("✅" if s["n"] == s["ids"] else "❌ duplicates"))
display(spark.sql(f"SELECT rank_in_city, restaurant_name, district, video_count, credited_views, credited_views_30d, round(like_rate, 4) AS like_rate, round(ranking_score, 3) AS score FROM {AGG} ORDER BY city, rank_in_city LIMIT 20"))
display(spark.sql(f"""
    SELECT legit_rank_in_city, restaurant_name, cuisine, price_band, google_rating, google_rating_count,
           google_rating_adjusted, channel_count, round(like_rate, 4) AS like_rate, legit_score, legit_label
    FROM {AGG} ORDER BY city, legit_rank_in_city LIMIT 20
"""))
display(spark.sql(f"SELECT legit_label, COUNT(*) AS n FROM {AGG} GROUP BY 1 ORDER BY 2 DESC"))
print("\n--- agg_restaurant_ranking complete ---")
